# Structured Output

- Models can be requested to provide their response in a format matching a given schema. 
- This is useful for ensuring the output can be easily parsed and used in subsequent processing. 
- LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [3]:
import os
from langchain.chat_models import init_chat_model
# print(os.environ.get("SSL_CERT_FILE"), os.environ.get("SSL_CERT_DIR"))

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model(model="groq:qwen/qwen3-32b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.12'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7fe17477ef30>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7fe112823ad0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [4]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="This year movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The movies rating out of 10")
    


In [5]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.12'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7fe17477ef30>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7fe112823ad0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'descr

In [6]:
response = model_with_structure.invoke("Provide details about the movie Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output alongside parsed structure

In [7]:
from pydantic import BaseModel, Field


class Movie(BaseModel):
    "A movie with details."
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="This year movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movies rating out of 10")


model_with_structure = model.with_structured_output(Movie, include_raw=True)
response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me check what tools I have available. There\'s a Movie function that requires title, year, director, and rating. I need to make sure I have all that information for Inception. \n\nFirst, the title is obviously "Inception". The year it was released was 2010. The director is Christopher Nolan. As for the rating, I think it\'s around 8.8 on IMDb. Let me confirm that. Yes, IMDb gives it 8.8/10. So all the required parameters are there. I\'ll structure the tool call with these details. Make sure the JSON is correctly formatted with the right data types: title and director as strings, year as an integer, and rating as a number. No need for any other functions since the user just wants the movie details. Alright, that should cover it.\n', 'tool_calls': [{'id': 'ymr9qt9ga', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inc

### Nested Structure

In [8]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor] = Field(default_factory=list)
    genres: list[str] = Field(default_factory=list)
    budget: float | None = Field(None, description="Budget in millions USD")


model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[], genres=[], budget=None)

## TypedDict
TypedDict provides a simpler alternative using python's built-in typing, ideal when you don't need runtime validation

In [9]:
from typing_extensions import TypedDict, Annotated


class MovieDict(TypedDict):
    """A movie with details"""
    "A movie with details."
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "This year movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[str, ..., "The movies rating out of 10"]


movie_with_typed_dict = model.with_structured_output(MovieDict)
response = movie_with_typed_dict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon',
 'rating': '8.0',
 'title': 'The Avengers',
 'year': 2012}

In [10]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor] = Field(default_factory=list)
    genres: list[str] = Field(default_factory=list)
    budget: float | None = Field(None, description="Budget in millions USD")


model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Please provide the details of the movie avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'}],
 'genres': ['Action', 'Science Fiction', 'Adventure'],
 'title': 'Avengers',
 'year': 2012}

In [13]:
model.profile # List of things model supports

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

In [14]:
import os 

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [18]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person"""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email of the person")
    phone: str = Field(description="The phone number of the person")


agent = create_agent(
    model="gpt-5.4-mini",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (123) 456-7890"}]
})

print(result["structured_response"])
result

name='John Doe' email='john@example.com' phone='(123) 456-7890'


{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (123) 456-7890', additional_kwargs={}, response_metadata={}, id='0c7e4d3f-3635-4aea-903a-6bfa25d8dd8f'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(123) 456-7890"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 203, 'total_tokens': 233, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-E1K8uW9d84VlvHm8IGcY7OnpyxTpf', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f5dc2-a0ed-7741-be8d-695f36bb0de9-0', tool_calls=[], invalid_tool_calls=[], usage_metada

In [19]:
## TypedDict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person"""
    name: str # The name of the person
    email: str # The email of the person
    phone: str #The  phone number of the person


agent = create_agent(
    model="gpt-5.4-mini",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (123) 456-7890"}]
})

print(result["structured_response"])
result

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(123) 456-7890'}


{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (123) 456-7890', additional_kwargs={}, response_metadata={}, id='2349cbfe-2dae-41ea-80ac-4c9a0dae5f11'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(123) 456-7890"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 178, 'total_tokens': 208, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-E1KCPvELMpUMXgmojAo5iiIzVRqUG', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f5dc5-efe4-7db0-bb8d-e22dbec5983d-0', tool_calls=[], invalid_tool_calls=[], usage_metada

## DataClasses

A data class is a class typically containing mainly daa, although threre aren't really any restrictions. You create it using the `@dataclass` decorator

In [20]:
from dataclasses import dataclass
from langchain.agents import create_agent


@dataclass
class ContactInfo:
    """Contact information for a person"""
    name: str # The name of the person
    email: str # The email of the person
    phone: str #The  phone number of the person


agent = create_agent(
    model="gpt-5.4-mini",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (123) 456-7890"}]
})

print(result["structured_response"])
result

ContactInfo(name='John Doe', email='john@example.com', phone='(123) 456-7890')


{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (123) 456-7890', additional_kwargs={}, response_metadata={}, id='9ec40ecf-9948-4184-8083-5da4f6f62e5e'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(123) 456-7890"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 178, 'total_tokens': 208, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-E1KEfFRtkvt9gj13DkPpgNMX0x42T', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f5dc8-1471-7410-9d46-80331b77184e-0', tool_calls=[], invalid_tool_calls=[], usage_metada